In [4]:
import requests
from datetime import datetime
import json
import statistics
usuarios = [
    {"nombre": "Ana", "rol": "admin", "edad": 34},
    {"nombre": "Luis", "rol": "editor", "edad": 28},
    {"nombre": "Marta", "rol": "lector", "edad": 41},
    {"nombre": "Juan", "rol": "admin", "edad": 22},
]

solo_admins = list(filter(lambda u: u["rol"] == "admin", usuarios))
nombres_admins = [u["nombre"] for u in solo_admins]
print(solo_admins)

[{'nombre': 'Ana', 'rol': 'admin', 'edad': 34}, {'nombre': 'Juan', 'rol': 'admin', 'edad': 22}]


In [5]:

class Usuario:
    def __init__(self,nombre,rol,edad):
      self.nombre=nombre
      self.rol=rol
      self.edad=edad

    def presentarse(self):
        return f"Soy {self.nombre} y mi rol es {self.rol}"

instancias = [
    Usuario(
        nombre=u["nombre"],
        rol=u["rol"],
        edad=u["edad"]
    )
    for u in usuarios
]
#instancias= [Usuario(**u) for u in usuarios]

for u in instancias:
    print(u.presentarse())

Soy Ana y mi rol es admin
Soy Luis y mi rol es editor
Soy Marta y mi rol es lector
Soy Juan y mi rol es admin


In [10]:

class UsuarioBasico(Usuario):
    def acciones_permitidas(self):
        return ["leer"]

class UsuarioEditor(Usuario):
    def acciones_permitidas(self):
        return ["leer", "editar"]

class UsuarioAdmin(Usuario):
    def acciones_permitidas(self):
        return ["leer", "editar", "eliminar"]

polimorfos= [
    UsuarioBasico("Marta", "lector", 41),
    UsuarioEditor("Luis", "editor", 28),
    UsuarioAdmin("Ana", "admin", 34),
]

for u in polimorfos:
    print(f"{u.nombre} ({u.rol}) → {u.acciones_permitidas()}")


Marta (lector) → ['leer']
Luis (editor) → ['leer', 'editar']
Ana (admin) → ['leer', 'editar', 'eliminar']


In [11]:
class PermisoDenegadoError(Exception):
    pass

def restringir_admin(func):
    def wrapper(usuario, *args, **kwargs):
        if getattr(usuario, "rol", None) != "admin":
            raise PermisoDenegadoError(f"Permiso denegado: {usuario.nombre} no es admin.")
        return func(usuario, *args, **kwargs)
    return wrapper

@restringir_admin
def borrar_usuario(usuario, objetivo):
    return f"{usuario.nombre} eliminó a {objetivo.nombre}"

admin = UsuarioAdmin("Ana", "admin", 34)
editor = UsuarioEditor("Luis", "editor", 28)

print(borrar_usuario(admin, Usuario("Juan", "admin", 22)))
try:
    print(borrar_usuario(editor, Usuario("Juan", "admin", 22)))
except PermisoDenegadoError as e:
    print("Excepción capturada:", e)


Ana eliminó a Juan
Excepción capturada: Permiso denegado: Luis no es admin.


In [12]:
def guardar_usuarios(usuarios, ruta= "usuarios.txt"):
    try:
        with open(ruta, "w", encoding="utf-8") as f:
            for u in usuarios:
                if isinstance(u, dict):
                    nombre, rol, edad = u["nombre"], u["rol"], u["edad"]
                else:
                    nombre, rol, edad = u.nombre, u.rol, u.edad
                f.write(f"{nombre},{rol},{edad}\n")
        print(f"Usuarios guardados en {ruta}")
        return True
    except OSError as e:
        print(f"Error al escribir el archivo '{ruta}': {e}")
        return False

guardar_usuarios(instancias, "usuarios.txt")

Usuarios guardados en usuarios.txt


True

In [22]:
OPEN_METEO_URL = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=-34.61&longitude=-58.38&hourly=temperature_2m"
)

def obtener_clima_y_guardar(ruta_json: str = "clima.json"):
    try:
        resp = requests.get(OPEN_METEO_URL, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        with open(ruta_json, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Clima guardado en {ruta_json} (Open-Meteo)")
        return data
    except (requests.RequestException, ValueError) as e:
    #olvidense de esta parte, es solo por si llegara a fallar la api
        print(f"Error al consultar Open-Meteo: {e}\nUsando datos simulados...")
        hoy = datetime.utcnow().date().isoformat()
        fallback = {
            "hourly": {
                "time": [f"{hoy}T{str(h).zfill(2)}:00" for h in range(24)],
                "temperature_2m": [18 + ((h * 7) % 13) for h in range(24)]
            }
        }
        with open(ruta_json, "w", encoding="utf-8") as f:
            json.dump(fallback, f, ensure_ascii=False, indent=2)
        print(f"Clima guardado en {ruta_json} (simulado)")
        return fallback

data_clima = obtener_clima_y_guardar("clima.json")

Clima guardado en clima.json (Open-Meteo)


In [16]:

def cargar_clima(ruta_json: str = "clima.json"):
    try:
        with open(ruta_json, "r", encoding="utf-8") as f:
            contenido = json.load(f)
        if "hourly" not in contenido or "temperature_2m" not in contenido["hourly"]:
            raise ValueError("Estructura JSON inesperada: falta 'hourly.temperature_2m'")
        return contenido
    except FileNotFoundError:
        print(f"❌ Archivo no encontrado: {ruta_json}")
        raise
    except json.JSONDecodeError as e:
        print(f"❌ Error de decodificación JSON: {e}")
        raise
    except ValueError as e:
        print(f"❌ Error de contenido: {e}")
        raise

print("\n# 2.2")
clima = cargar_clima("clima.json")
temps = clima["hourly"]["temperature_2m"]
horas = clima["hourly"]["time"]
print(f"Leídas {len(temps)} temperaturas.")
print("Primeras 5:", list(zip(horas, temps))[:5])




# 2.2
Leídas 168 temperaturas.
Primeras 5: [('2025-10-11T00:00', 21.5), ('2025-10-11T01:00', 20.8), ('2025-10-11T02:00', 20.2), ('2025-10-11T03:00', 19.7), ('2025-10-11T04:00', 19.2)]


In [19]:
def filtrar_calor(temperaturas, umbral):
    for t in temperaturas:
        if t > umbral:
            yield t

mayores_25 = list(filtrar_calor(temps, umbral=25))
print(f"Temperaturas > 25°C: {len(mayores_25)}")
print("Muestras:", mayores_25[:10])



Temperaturas > 25°C: 27
Muestras: [25.7, 26.7, 27.2, 27.4, 26.8, 25.5, 25.9, 26.9, 27.2, 27.2]


In [20]:
redondear = lambda x: int(round(x))
dicc_10 = {
    horas[i]: redondear(temps[i])
    for i in range(min(10, len(temps)))
}

print("\n# 2.4")
print("Diccionario hora→temp (10 primeras):")
for k, v in dicc_10.items():
    print(k, "→", v, "°C")


# 2.4
Diccionario hora→temp (10 primeras):
2025-10-11T00:00 → 22 °C
2025-10-11T01:00 → 21 °C
2025-10-11T02:00 → 20 °C
2025-10-11T03:00 → 20 °C
2025-10-11T04:00 → 19 °C
2025-10-11T05:00 → 19 °C
2025-10-11T06:00 → 18 °C
2025-10-11T07:00 → 18 °C
2025-10-11T08:00 → 18 °C
2025-10-11T09:00 → 17 °C


In [21]:

def formatear_salida(func):
    def wrapper(self, *args, **kwargs):
        print("=" * 40)
        print(f"   Reporte Climático – {self.fecha}")
        print("=" * 40)
        return func(self, *args, **kwargs)
    return wrapper

class ReporteClimatico:
    def __init__(self, fecha, temperaturas):
        self.fecha = fecha
        self.temperaturas = list(temperaturas)

    @formatear_salida
    def resumen(self):
        tmax = max(self.temperaturas)
        tmin = min(self.temperaturas)
        tprom = statistics.fmean(self.temperaturas)
        print(f"Máxima:  {tmax:.2f} °C")
        print(f"Mínima:  {tmin:.2f} °C")
        print(f"Promedio:{tprom:.2f} °C")
        return {"max": tmax, "min": tmin, "promedio": tprom}

fecha_rep = horas[0].split("T")[0] if horas else datetime.utcnow().date().isoformat()
reporte = ReporteClimatico(fecha=fecha_rep, temperaturas=temps)
_ = reporte.resumen()

   Reporte Climático – 2025-10-11
Máxima:  28.60 °C
Mínima:  13.70 °C
Promedio:19.91 °C
